<div style="border-left: 5px solid #b7791f; background-color: #fff8e1; padding: 0.8em 1em; margin: 1em 0; border-radius: 4px;">
  <strong>Warning: AI-assisted materials</strong><br><br>
  These materials were developed with assistance from AI tools. All content has been reviewed and edited by the instructor, who takes final responsibility for its accuracy, clarity, and appropriateness for the course. Students should treat these materials as instructor-reviewed course content while applying the same critical judgment they would use with any technical material. Please report any suspected errors or unclear explanations to ghunt@wm.edu.
</div>

# Logistic Regression

We now consider a binary outcome, encoded as
$$
y \in \{0,1\}
$$
where 0 and 1 are class labels rather than measurements on a meaningful numerical scale. This encoding is conventional: reversing the labels would reverse which direction of the score favors each class, but would not change the kind of problem being modeled. As before, $x\in\mathbb{R}^D$.

Following the ERM framework, we first choose a model class. We use a score that is **linear in the parameters**:
$$
s_w(x) = w^\top x.
$$
When the dependence on $w$ is clear, we write $s(x)$ for $s_w(x)$. Higher scores favor class 1 and lower scores favor class 0. Because the score can be any real number, it is not itself a probability. To map it into $(0,1)$, define

$$
p_w(x)=\sigma(s_w(x)),
\qquad
\sigma(t)=\frac{1}{1+e^{-t}}.
$$

The sigmoid is increasing, with

- if $w^\top x \gg 0$, then $\sigma(w^\top x) \approx 1$
- if $w^\top x \ll 0$, then $\sigma(w^\top x) \approx 0$
- if $w^\top x = 0$, then $\sigma(w^\top x) = 1/2$

At a score of zero the two classes receive equal support; the sign determines which class is favored, and the magnitude measures how strongly. Thus $w^\top x$ does the linear modeling, while $p_w(x)$ is interpreted as the predicted probability of class 1.

## Cross-Entropy Loss

We want small loss when $p$ agrees with the observed class and large loss when it does not. The **cross-entropy loss** as a function of the predicted probability is
$$
L_{\mathrm{CE}}(y,p)=-\Big(y\log p+(1-y)\log(1-p)\Big).
$$
As a function of the score $s$, this becomes
$$
\ell(y,s)=L_{\mathrm{CE}}(y,\sigma(s))
=\log(1+e^s)-ys.
$$
The probability form is useful for interpretation, while the score form is often easier to differentiate and optimize.

### Understanding the loss

For the two possible labels,
$$
L_{\mathrm{CE}}(1,p)=-\log p,
\qquad
L_{\mathrm{CE}}(0,p)=-\log(1-p).
$$
When $y=1$, the loss approaches zero as $p\to1$ and diverges as $p\to0$; when $y=0$, the roles are reversed. At $p=1/2$, either class gives loss $\log 2$. Thus the loss is small for correct, confident predictions and grows without bound as the model becomes confidently wrong.

### Why not squared error?

Squared error on the predicted probability is a valid alternative:
$$
L_{\mathrm{SE}}(y,p)=(y-p)^2.
$$
Cross-entropy penalizes confident mistakes more strongly and, with a linear score and sigmoid, gives a convex objective in $w$; the squared-error composition generally does not. The following plot compares them:

In [ ]:
#| code-fold: true

import numpy as np
import matplotlib.pyplot as plt

def cross_entropy(y, p):
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))

def squared_error(y, p):
    return (y - p) ** 2

# values of p = sigma(s)
p_vals = np.linspace(0.001, 0.999, 200)

# compute losses
ce_y1 = cross_entropy(1, p_vals)
se_y1 = squared_error(1, p_vals)

ce_y0 = cross_entropy(0, p_vals)
se_y0 = squared_error(0, p_vals)

# plot
plt.figure()

# y = 1
plt.plot(p_vals, ce_y1, label="CE (y=1)")
plt.plot(p_vals, se_y1, linestyle='--', label="SE (y=1)")

# y = 0
plt.plot(p_vals, ce_y0, label="CE (y=0)")
plt.plot(p_vals, se_y0, linestyle='--', label="SE (y=0)")

plt.xlabel("p = σ(s)")
plt.ylabel("Loss")
plt.title("Cross-Entropy vs Squared Error")
plt.legend()
plt.grid(True)

plt.show()

### ERM

For observations $(x_1,y_1),\ldots,(x_N,y_N)$, the empirical risk is the average cross-entropy loss:
$$
\begin{aligned}
\hat{R}(w)=\frac{1}{N}\sum_{n=1}^N
\left[\log(1+e^{w^\top x_n})-y_nw^\top x_n\right]
&= \frac{1}{N}\sum_{n=1}^N-\Big(y_n \log p_n + (1-y_n)\log(1-p_n)\Big)
\end{aligned}
$$
ERM chooses
$$
\hat{w}=\arg\min_w\hat{R}(w).
$$
The factor $1/N$ does not affect the minimizer and is sometimes omitted. This has the same structure as least squares: choose a score class, choose a loss, and minimize the resulting empirical risk. Only the loss and optimization method have changed.


## Vector Form

Let $X\in\mathbb{R}^{N\times D}$ have row $n$ equal to $x_n^\top$, and let
$$
y=(y_1,\dots,y_N)^\top.
$$
The score and probability vectors are
$$
Xw\in\mathbb{R}^N,
\qquad
p=\sigma(Xw),
$$
where the sigmoid is applied elementwise, so $p_n=\sigma(x_n^\top w)$. Both vectors have length $N$. 



## No Closed-Form Solution

Unlike the least-squares objective, $\hat R(w)$ is not quadratic in $w$. Setting its gradient equal to zero produces nonlinear equations in $w$ that cannot be rearranged into an analogous closed-form expression for $\hat w$. We therefore use an iterative optimization method such as gradient descent.

The absence of a closed form does not make the problem poorly behaved: the logistic-regression objective is convex in $w$. 


## Gradient Descent

The gradient $\nabla \hat R(w)$ describes how the objective changes locally: the gradient points in the direction of steepest increase, so its negative points in the direction of steepest decrease. Starting from an initial value $w^{(0)}$, gradient descent repeats
$$
w^{(t+1)}=w^{(t)}-\eta\nabla\hat R(w^{(t)}),
$$
where the gradient is evaluated at the current parameter vector $w^{(t)}$. The update takes a step downhill to produce $w^{(t+1)}$, then recomputes the gradient at that new point. Repeating these local steps traces a path across the objective surface toward a minimizer rather than solving for it in one algebraic step.

The learning rate $\eta>0$ controls the step size. If it is too small, progress is slow; if it is too large, an update can overshoot the low part of the surface and the algorithm may fail to converge. In practice, iterations continue until the objective, parameters, or gradient change very little. The remaining task is to compute $\nabla\hat R(w)$ for logistic regression.


The sigmoid derivative is
$$
\frac{d\sigma(t)}{dt}=\sigma(t)(1-\sigma(t)).
$$

Now let
$$
p_n = \sigma(x_n^\top w).
$$

For the $n$th loss,
$$
\ell_n = -\Big(y_n \log p_n + (1-y_n)\log(1-p_n)\Big),
$$

One can show that 

$$
\nabla_w\ell_n= (p_n-y_n)x_n.
$$

Summing over all $n$, we obtain

$$
\nabla \hat{R}(w) = \frac{1}{N}\sum_{n=1}^N (p_n-y_n)x_n.
$$

In matrix form, this is

$$
\nabla \hat{R}(w)=\frac{1}{N}X^\top (p-y).
$$

For comparison, least squares uses the residual $Xw-y$; logistic regression uses the probability residual $p-y$.



Consequently, gradient descent takes the form

$$
w^{(t+1)}=w^{(t)} - \eta \nabla \hat{R}(w^{(t)})=w^{(t)} - \eta \frac{1}{N}X^\top(p^{(t)}-y),
$$
where $\eta > 0$ is the learning rate.

So each iteration is:

1. compute the current "logits" $Xw^{(t)}$
2. apply the sigmoid to get $p^{(t)}$
3. compute the gradient $\frac{1}{N}X^\top(p^{(t)}-y)$
4. update $w$


Let's try implementing this in code:

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def cross_entropy_loss(X, y, w):
    p = sigmoid(X @ w)

    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def logistic_gradient(X, y, w):
    p = sigmoid(X @ w)
    return (X.T @ (p - y)) / len(y)

def gradient_descent_with_path(X, y, w0, lr=0.1, n_steps=1000, tol=1e-6):
    w = w0.copy()
    path = [w.copy()]
    losses = [cross_entropy_loss(X, y, w)]

    for _ in range(n_steps):
        grad = logistic_gradient(X, y, w)
        w_new = w - lr * grad

        # stopping condition
        if np.linalg.norm(w_new - w) < tol:
            w = w_new
            path.append(w.copy())
            losses.append(cross_entropy_loss(X, y, w))
            break

        w = w_new
        path.append(w.copy())
        losses.append(cross_entropy_loss(X, y, w))

    return w, np.array(losses), np.array(path)

Let's generate some really simple data:

In [ ]:
np.random.seed(12390)

n = 100

# Class 1
X_pos = np.random.randn(n//2, 2) + np.array([1.0, 1.0])

# Class 0
X_neg = np.random.randn(n//2, 2) + np.array([-1.0, -1.0])

# Stack together
X = np.vstack([X_pos, X_neg])
y = np.hstack([np.ones(n//2), np.zeros(n//2)])

print(X[:10])
print(y[45:55])

In [ ]:
plt.figure(figsize=(5,5))

plt.scatter(X_pos[:,0], X_pos[:,1], label="y = 1")
plt.scatter(X_neg[:,0], X_neg[:,1], label="y = 0")

plt.xlabel(r"$x_1$")
plt.ylabel(r"$x_2$")
plt.title("2D dataset")
plt.legend()
plt.grid(True)
plt.axis("equal")

plt.show()

Now we can run the gradient descent:

In [ ]:
w0 = np.array([0.0, 0.0])
w_hat, losses, path = gradient_descent_with_path(X, y, w0, lr=0.1, n_steps=10000, tol=1e-4)

In [ ]:
print("Final weights:", w_hat)
print("Final loss:", losses[-1])
print("Actual num steps:",len(path))

We can plot the loss v. iteration:

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(losses, marker="o", markersize=3)
plt.xlabel("Iteration")
plt.ylabel("Cross-entropy loss")
plt.title("Loss during gradient descent")
plt.grid(True)
plt.show()

We can also view the optimization path in parameter space. Each point is one gradient-descent iterate; the contours show values of the same cross-entropy objective.

In [ ]:
w1_vals = np.linspace(-3, 3, 200)
w2_vals = np.linspace(-3, 3, 200)
W1, W2 = np.meshgrid(w1_vals, w2_vals)

Z = np.zeros_like(W1)
for i in range(W1.shape[0]):
    for j in range(W1.shape[1]):
        w = np.array([W1[i, j], W2[i, j]])
        Z[i, j] = cross_entropy_loss(X, y, w)

plt.figure(figsize=(7, 6))
contours = plt.contour(W1, W2, Z, levels=25)
plt.clabel(contours, inline=True, fontsize=8)
plt.plot(path[:, 0], path[:, 1], marker="o", markersize=3)
plt.scatter(path[0, 0], path[0, 1], s=80, marker="x", label="start")
plt.scatter(path[-1, 0], path[-1, 1], s=80, marker="*", label="end")
plt.xlabel(r"$w_1$")
plt.ylabel(r"$w_2$")
plt.title("Loss surface and gradient-descent path")
plt.legend()
plt.grid(True)
plt.show()

## From Scores to Predictions

The ERM problem above selects $\hat w$ by minimizing cross-entropy, but the fitted model first gives us a score $s_{\hat w}(x)$ and a probability $p_{\hat w}(x)$, not a hard class label. As in the ERM framework from the introductory lecture, we still need an **action** that converts the fitted score into a prediction.

One way to describe a binary action is to use two class scores,
$$
s_k(x)=w_k^\top x,
\qquad
\hat y=\arg\max_{k=0,1}s_k(x).
$$
The two scores are actually redundant: only their difference affects the prediction. In particular: 
$$
\hat y=1
\quad\Longleftrightarrow\quad s_1(x) \geq s_0(x) \quad\Longleftrightarrow\quad s_1(x)-s_0(x) \geq 0 \quad\Longleftrightarrow\quad
s_1(x)-s_0(x)=(w_1-w_0)^\top x\ge0.
$$
Defining $w=w_1-w_0$ gives the single score $s_w(x)=w^\top x$ used in our ERM problem; it represents how much the model favors class 1 over class 0. Equivalently, class 0 can be treated as the **reference class** by setting $s_0(x)=0$ and $s_1(x)=w^\top x$. After fitting $\hat w$, either representation gives the prediction rule
$$
\hat y=
\begin{cases}
1 & \text{if } \hat w^\top x\ge0,\\
0 & \text{otherwise}.
\end{cases}
$$
Because $\sigma$ is increasing and $\sigma(0)=1/2$, thresholding the score at zero is equivalent to thresholding the predicted probability at one-half:
$$
\hat y=1\quad\text{if }p_{\hat w}(x)\ge\frac12.
$$

### Decision Boundary

The **decision boundary** is where the predicted class changes:
$$
\hat{w}^\top x = 0.
$$
It is linear in the chosen features.


We can plot the decision boundary learned from the previous problem:

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(X_pos[:, 0], X_pos[:, 1], label="y = 1")
plt.scatter(X_neg[:, 0], X_neg[:, 1], label="y = 0")

# Decision boundary: w1 x1 + w2 x2 = 0
if abs(w_hat[1]) > 1e-12:
    x1_vals = np.array([X[:, 0].min() - 1, X[:, 0].max() + 1])
    x2_vals = -(w_hat[0] / w_hat[1]) * x1_vals
    plt.plot(x1_vals, x2_vals, label=r"$w^\top x = 0$")
else:
    x_const = 0.0
    if abs(w_hat[0]) > 1e-12:
        x_const = 0.0
    plt.axvline(x=x_const, label=r"$w^\top x = 0$")

plt.xlabel(r"$x_1$")
plt.ylabel(r"$x_2$")
plt.title("Data and learned decision boundary")
plt.legend()
plt.grid(True)
plt.axis("equal")
plt.show()


## Feature Design and Limitations

The columns of $X$ may be measured covariates or engineered features such as powers, interactions, or transformations of the original variables. Logistic regression remains linear in $w$ and has a linear boundary in the chosen feature space, but the boundary need not be linear in the raw inputs. For example, including $x_1^2$ or $x_1x_2$ can produce a curved boundary in the original input space.

This is both useful and limiting: the model can represent nonlinear boundaries through feature engineering, but it cannot represent a complicated boundary unless the chosen features make that boundary accessible.


### Perfect Separation

Data are **perfectly linearly separable** if some $w$ satisfies
$$
y_n = 1 \implies w^\top x_n > 0, \quad
y_n = 0 \implies w^\top x_n < 0.
$$
Then $cw$ gives the same classifications for every $c>0$, while the cross-entropy loss approaches zero as $c\to\infty$. Consequently, the unregularized objective has no finite minimizer: an optimizer may return very large coefficients or issue a warning. Regularization, introduced later, prevents this behavior.

## Code Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.inspection import DecisionBoundaryDisplay

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (6, 4)

### Penguins

We use the [penguins](https://allisonhorst.github.io/palmerpenguins/) dataset, loaded from a CSV:

In [ ]:
url = "https://gist.githubusercontent.com/slopp/ce3b90b9168f2f921784de84fa445651/raw/penguins.csv"
penguins = pd.read_csv(url)
# clean up and shuffle
penguins = penguins.dropna().copy().sample(frac=1, random_state=4222)

# for ease of visualization, let's subset down to two vars + species
cols = ["flipper_length_mm", "bill_depth_mm", "species"]
penguins = penguins[cols]
penguins.head()

**Binary logistic regression: Adelie vs Chinstrap**

First, retain the Adelie and Chinstrap observations:

In [ ]:
# subset down to only Adelie and Chinstrap
d = penguins.loc[penguins["species"].isin(["Adelie", "Chinstrap"]), cols].copy()

d["species"].value_counts()

In [ ]:
fig, ax = plt.subplots()
sns.scatterplot(
    data=d,
    x="flipper_length_mm",
    y="bill_depth_mm",
    hue="species",
    ax=ax,
)
ax.set_title("Adelie vs Chinstrap")
plt.show()

**Fit a logistic regression model**

Using flipper length and bill depth, we predict Chinstrap ($y=1$) versus Adelie ($y=0$). Scikit-learn orders the class labels alphabetically.

In [ ]:
X_train = d[["flipper_length_mm","bill_depth_mm"]]

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train))
X_train_scaled.columns = ["flipper_length_mm","bill_depth_mm"]

y_train = d["species"]

In [ ]:
log_mod = LogisticRegression(C=np.inf)
log_mod.fit(X_train_scaled, y_train)

Setting `C=np.inf` removes "regularization"; we return to this parameter later.

**Scores, probabilities, and predictions**

- `decision_function` gives the linear score
- `predict_proba` gives class probabilities
- `predict` thresholds those probabilities at 0.5 in the binary case

In [ ]:
scores = log_mod.decision_function(X_train_scaled)
scores[:10]

In [ ]:
probs = log_mod.predict_proba(X_train_scaled)
probs[:10]

In [ ]:
preds = log_mod.predict(X_train_scaled)
preds[:10]

**Confusion matrix**

A confusion matrix compares true labels (rows) with predicted labels (columns):
$$
\begin{array}{c|cc}
 & \text{Pred 0} & \text{Pred 1} \\
\hline
\text{True 0} & \text{TN} & \text{FP} \\
\text{True 1} & \text{FN} & \text{TP}
\end{array}
$$
TP and TN are correct predictions. FP is a false alarm for class 1, while FN is a missed class-1 observation. Unlike accuracy alone, the table distinguishes these two kinds of error, which can matter when classes are imbalanced or the errors have different costs.

In [ ]:
confusion_matrix(y_train, preds)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_train, preds),
    display_labels=["Adelie", "Chinstrap"],
).plot(ax=ax, colorbar=False)
ax.grid(False)
plt.show()

**Precision, recall, and F1 score**

$$
\text{Precision}=\frac{\text{TP}}{\text{TP}+\text{FP}},
\qquad
\text{Recall}=\frac{\text{TP}}{\text{TP}+\text{FN}}.
$$
Precision asks: among observations predicted as class 1, what fraction are actually class 1? High precision means few false positives. Recall asks: among the actual class-1 observations, what fraction did we find? High recall means few false negatives. Their harmonic mean is
$$
\text{F1}=2\frac{\text{Precision}\cdot\text{Recall}}
{\text{Precision}+\text{Recall}}.
$$
F1 is high only when both precision and recall are high. 

Scikit-learn provides these metrics through `classification_report`:

In [ ]:
print(classification_report(y_train, preds, target_names=["Adelie", "Chinstrap"]))

Our `gradient_descent_with_path` implementation should give comparable coefficients:

In [ ]:
X_design = np.column_stack([np.ones(len(X_train_scaled)), X_train_scaled])
y01 = (y_train=="Chinstrap")*1 #Adelie = 0 (base), Chinstrap=1

In [ ]:
out = gradient_descent_with_path(X_design,y01,w0=np.random.normal(0,1,3),lr=1E-2, n_steps=10000)

In [ ]:
out[0]

The scikit-learn coefficients are:

In [ ]:
print(log_mod.intercept_)
print(log_mod.coef_)

**Decision regions**

In [ ]:
def plot_binary_boundary(model, X, y, title, ax=None, palette = {
        "Adelie": "#1f77b4",
        "Chinstrap": "#ff7f0e",
    }):
    if ax is None:
        fig, ax = plt.subplots()

    plot_df = X.reset_index(drop=True).copy()
    plot_df["species"] = y.reset_index(drop=True)

    from matplotlib.colors import ListedColormap
    cmap = ListedColormap([palette[c] for c in model.classes_])

    DecisionBoundaryDisplay.from_estimator(
        model,
        X,
        response_method="predict",
        cmap=cmap,
        alpha=0.25,
        ax=ax,
        eps=0.1,
        grid_resolution=300
    )

    sns.scatterplot(
        data=plot_df,
        x="flipper_length_mm",
        y="bill_depth_mm",
        hue="species",
        palette=palette,
        ax=ax,
        edgecolor="black",
        s=60,
    )

    ax.set_title(title)
    return ax

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
plot_binary_boundary(log_mod, X_train_scaled, y_train, "Linear logistic regression", ax=ax)
plt.show()

Adding second-degree polynomial features allows a nonlinear boundary in the raw inputs:

In [ ]:
quad_mod = make_pipeline(
    StandardScaler(),
    PolynomialFeatures(degree=2, include_bias=False),
    LogisticRegression(C=np.inf)
)
quad_mod.fit(X_train, y_train)

The resulting boundary is curved in the raw feature space:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
plot_binary_boundary(quad_mod, X_train, y_train, "Quadratic logistic regression", ax=ax)
plt.show()

### A Separable Example

For this sample, Adelie and Gentoo penguins are linearly separable using these two features:

In [ ]:
d_sep = penguins.loc[penguins["species"].isin(["Adelie", "Gentoo"]), cols].copy()

In [ ]:
X_sep = d_sep[["flipper_length_mm", "bill_depth_mm"]]
y_sep = d_sep["species"]

In [ ]:
sep_mod = LogisticRegression(C=np.inf)
sep_mod.fit(X_sep, y_sep)

In [ ]:
pal = {
        "Adelie": "#1f77b4",
        "Gentoo": "#ff7f0e",
    }
fig, ax = plt.subplots(figsize=(6, 5))
plot_binary_boundary(sep_mod, X_sep, y_sep, "Linear logistic regression", ax=ax, palette=pal)
plt.show()

Scikit-learn returns finite coefficients because its numerical optimizer stops after reaching a tolerance, not because the unregularized objective has a finite minimizer. Near-perfect training accuracy and probabilities close to 0 or 1 are warning signs.

In [ ]:
sep_mod.predict_proba(X_sep)[:10]

## Review Questions

See: @sec-logistic-questions.